In [ ]:
import os
import logging

# Suppress TensorFlow C++ backend warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 

import tensorflow as tf

# Suppress TensorFlow Python backend warnings
tf.get_logger().setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

import shutil
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# UNIFIED IMPORT ENGINE: Use pure tf.keras directly to prevent serialization mismatches
from tensorflow.keras import Input
from tensorflow.keras.layers import (Dense, GlobalAveragePooling2D, Dropout,
                                     BatchNormalization, Conv2D, Reshape,
                                     UpSampling2D, Concatenate, AveragePooling2D, Layer, Flatten)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print(f"TensorFlow Engine Loaded Successfully: Version {tf.__version__}")

# ─────────────────────────────────────────────────────────────────────────
#  KAGGLE DATASET PATH ROUTING
# ─────────────────────────────────────────────────────────────────────────
print("Routing Kaggle mounted dataset paths...")

DATASET_DIR = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
TRAIN_DIR = os.path.join(DATASET_DIR, "Training")
TEST_DIR  = os.path.join(DATASET_DIR, "Testing")

# ─────────────────────────────────────────────────────────────────────────
#  CONFIG & HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────
IMG_SIZE        = (224, 224)
BATCH_SIZE      = 32
EPOCHS_FROZEN   = 10
EPOCHS_FINETUNE = 15
LEARNING_RATE   = 1e-4
FINE_TUNE_LR    = 4e-5
NUM_CLASSES     = 4
SEED            = 42
VAL_SPLIT       = 0.20

N_QUBITS        = 4
N_LAYERS        = 3

# ─────────────────────────────────────────────────────────────────────────
#  OPTIMIZED TENSORFLOW QUANTUM LAYER SIMULATOR
# ─────────────────────────────────────────────────────────────────────────
@tf.keras.utils.register_keras_serializable()
class TFQuantumSimulationLayer(Layer):
    """
    A purely vectorized, batch-optimized TensorFlow quantum circuit simulator.
    Utilizes tf.einsum for parallel tensor contractions and precomputed permutations
    to bypass tf.vectorized_map bottlenecks.
    """
    def __init__(self, n_qubits=4, n_layers=3, **kwargs) -> None:
        super().__init__(**kwargs)
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.state_size = 2 ** n_qubits
        
        # Precompute CNOT permutations for adjacent qubits to avoid loops during forward pass
        self.cnot_perms = []
        for q in range(self.n_qubits - 1):
            self.cnot_perms.append(self._get_cnot_perm(control=q, target=q+1))
            
        # Precompute Z masks for expectation values (Observables)
        self.z_masks = []
        for q in range(self.n_qubits):
            mask = []
            for idx in range(self.state_size):
                bit_active = (idx & (1 << (self.n_qubits - 1 - q))) != 0
                mask.append(-1.0 if bit_active else 1.0)
            self.z_masks.append(tf.constant(mask, dtype=tf.float32))

    def _get_cnot_perm(self, control, target):
        perm = []
        for i in range(self.state_size):
            if (i & (1 << (self.n_qubits - 1 - control))):
                perm.append(i ^ (1 << (self.n_qubits - 1 - target)))
            else:
                perm.append(i)
        return tf.constant(perm, dtype=tf.int32)

    def build(self, input_shape) -> None:
        self.params = self.add_weight(
            name="quantum_weights",
            shape=(self.n_layers, self.n_qubits, 3),
            initializer="random_uniform",
            trainable=True
        )
        super().build(input_shape)

    def _get_rotation_matrices(self, phi, theta, omega):
        # Calculate rotation components across the entire batch simultaneously
        phi = tf.cast(phi, tf.complex64)
        theta = tf.cast(theta, tf.complex64)
        omega = tf.cast(omega, tf.complex64)

        cos_t = tf.cos(theta / 2.0)
        sin_t = tf.sin(theta / 2.0)

        u00 = tf.exp(-1j * (phi + omega) / 2.0) * cos_t
        u01 = -tf.exp(-1j * (phi - omega) / 2.0) * sin_t
        u10 = tf.exp(1j * (phi - omega) / 2.0) * sin_t
        u11 = tf.exp(1j * (phi + omega) / 2.0) * cos_t

        row0 = tf.stack([u00, u01], axis=-1)
        row1 = tf.stack([u10, u11], axis=-1)
        
        return tf.stack([row0, row1], axis=-2)

    def call(self, inputs):
        x = tf.clip_by_value(inputs * np.pi, -np.pi, np.pi)
        orig_shape = tf.shape(x)
        
        batch_flat = tf.reshape(x, (-1, self.n_qubits))
        batch_size = tf.shape(batch_flat)[0]

        state = tf.concat([
            tf.ones((batch_size, 1), dtype=tf.complex64), 
            tf.zeros((batch_size, self.state_size - 1), dtype=tf.complex64)
        ], axis=1)

        def apply_gate(st, U, qubit):
            st_reshaped = tf.reshape(st, [-1, 2**qubit, 2, 2**(self.n_qubits - 1 - qubit)])
            new_st = tf.einsum('boi, blir -> blor', U, st_reshaped)
            return tf.reshape(new_st, [-1, self.state_size])

        for l in range(self.n_layers):
            for q in range(self.n_qubits):
                U = self._get_rotation_matrices(batch_flat[:, q], batch_flat[:, q], batch_flat[:, q])
                state = apply_gate(state, U, q)
                
            for q in range(self.n_qubits):
                p = self.params[l, q] 
                
                p_phi = p[0] * tf.ones([batch_size], dtype=tf.float32)
                p_theta = p[1] * tf.ones([batch_size], dtype=tf.float32)
                p_omega = p[2] * tf.ones([batch_size], dtype=tf.float32)
                
                U = self._get_rotation_matrices(p_phi, p_theta, p_omega)
                state = apply_gate(state, U, q)
                
            for q in range(self.n_qubits - 1):
                state = tf.gather(state, self.cnot_perms[q], axis=1)

        state_density = tf.abs(state) ** 2
        
        expectations = []
        for q in range(self.n_qubits):
            exp_val = tf.reduce_sum(state_density * self.z_masks[q], axis=1)
            expectations.append(exp_val)
            
        quantum_out = tf.stack(expectations, axis=1)
        return tf.reshape(quantum_out, (orig_shape[0], orig_shape[1], self.n_qubits))

    def get_config(self):
        config = super().get_config()
        config.update({"n_qubits": self.n_qubits, "n_layers": self.n_layers})
        return config

# ─────────────────────────────────────────────────────────────────────────
#  CUSTOM SERIALIZATION CALLBACK SAFEGUARD
# ─────────────────────────────────────────────────────────────────────────
class SafeModelCheckpoint(Callback):
    """Saves model weights exclusively to avoid JSON layer structure translation bugs."""
    def __init__(self, filepath, monitor='val_accuracy'):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.best = -float('inf') if 'accuracy' in monitor else float('inf')

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is not None:
            is_better = (current > self.best) if 'accuracy' in self.monitor else (current < self.best)
            if is_better:
                self.best = current
                self.model.save_weights(self.filepath)
                print(f"\n[Saved] Weights improved to {current:.4f}. File updated.")

# ─────────────────────────────────────────────────────────────────────────
#  DATA GENERATORS SETUP (Updated with preprocess_input)
# ─────────────────────────────────────────────────────────────────────────
print("\nPreparing Data Flows...")
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10, width_shift_range=0.05,
    height_shift_range=0.05, zoom_range=0.05, brightness_range=[0.9, 1.1],
    horizontal_flip=True, validation_split=VAL_SPLIT,
)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="training", seed=SEED, shuffle=True,
)
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="validation", seed=SEED, shuffle=False,
)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False,
)

CLASS_NAMES = list(train_gen.class_indices.keys())
labels = train_gen.classes
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(class_weights))

print(f"\nClasses Detected  : {CLASS_NAMES}")
print(f"Training Samples  : {train_gen.samples}")
print(f"Validation Samples: {val_gen.samples}")
print(f"Test Samples      : {test_gen.samples}\n")

# ─────────────────────────────────────────────────────────────────────────
#  BUILD PARALLEL HYBRID MODEL
# ─────────────────────────────────────────────────────────────────────────
def build_hybrid_model(num_classes, trainable_base=False):
    img_h, img_w = int(IMG_SIZE[0]), int(IMG_SIZE[1])
    inputs = Input(shape=(img_h, img_w, 3), name="model_input")

    # ─── BRANCH 1: Classical Powerhouse (MobileNetV2) ───
    base_model = tf.keras.applications.MobileNetV2(
        weights="imagenet", include_top=False, input_shape=(img_h, img_w, 3)
    )
    base_model.trainable = trainable_base
    
    # CRITICAL: training=False forces BatchNorm layers to stay frozen during Phase 2
    x_mb = base_model(inputs, training=False) 
    x_mb = GlobalAveragePooling2D(name="classical_global_pool")(x_mb)

    # ─── BRANCH 2: Quantum Feature Extractor ───
    # Shrink the image to a manageable size for the 4-qubit simulator
    x_q = AveragePooling2D(pool_size=56, strides=56, name="q_downsample")(inputs)
    x_q = Conv2D(N_QUBITS, kernel_size=1, activation="relu", name="q_channels")(x_q)
    x_q = Reshape((-1, N_QUBITS), name="q_reshape")(x_q)
    
    x_q = TFQuantumSimulationLayer(N_QUBITS, N_LAYERS, name="quantum_simulator")(x_q)
    
    x_q = Flatten(name="q_flatten")(x_q)
    x_q = Dense(32, activation="relu", name="q_dense_proj")(x_q)

    # ─── FUSION & CLASSIFICATION HEAD ───
    x_fused = Concatenate(name="hybrid_fusion")([x_mb, x_q])
    
    x = Dense(512, activation="relu")(x_fused)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x) 
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(inputs=inputs, outputs=outputs), base_model

model, base_model = build_hybrid_model(NUM_CLASSES, trainable_base=False)
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# ─────────────────────────────────────────────────────────────────────────
#  TRAINING PIPELINE SUBMISSION
# ─────────────────────────────────────────────────────────────────────────
callbacks_frozen = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7, verbose=1),
    SafeModelCheckpoint("best_hybrid_frozen.weights.h5", monitor="val_accuracy"),
]

callbacks_finetune = [
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-8, verbose=1),
    SafeModelCheckpoint("best_hybrid_finetuned.weights.h5", monitor="val_accuracy"),
]

print("="*70)
print(" STARTING PHASE 1: Training Classification Head + Quantum Simulation Layer")
print("="*70)
history_frozen = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_FROZEN, callbacks=callbacks_frozen,
    class_weight=class_weight_dict,
)

print("\n" + "="*70)
print(" STARTING PHASE 2: Fine-Tuning Last 90 Layers of MobileNetV2 Base Backbone")
print("="*70)
base_model.trainable = True
for layer in base_model.layers[:-90]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_finetune = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_FINETUNE, callbacks=callbacks_finetune,
    class_weight=class_weight_dict,
)

# ─────────────────────────────────────────────────────────────────────────
#  EVALUATION METRICS & PLOTTING VISUALIZATIONS
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print(" EVALUATING ON THE UNSEEN TESTING KAGGLE DATASET")
print("="*70)

test_loss, test_accuracy = model.evaluate(test_gen, verbose=1)
print(f"\nFinal Test Set Loss    : {test_loss:.4f}")
print(f"Final Test Set Accuracy: {test_accuracy * 100:.2f}%")

test_gen.reset()
y_pred_probs = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

print("\nDetailed Performance Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

def merge_history(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

full_history = merge_history(history_frozen, history_finetune)
epochs_range = range(1, len(full_history["accuracy"]) + 1)
phase1_end   = len(history_frozen.history["accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Parallel Hybrid QCNN + MobileNetV2 History", fontsize=12, fontweight="bold")

for ax, (tk, vk), title in zip(axes, [("accuracy", "val_accuracy"), ("loss", "val_loss")], ["Accuracy", "Loss"]):
    ax.plot(epochs_range, full_history[tk], label=f"Train {title}", color="steelblue", lw=2)
    ax.plot(epochs_range, full_history[vk], label=f"Val {title}",   color="darkorange", lw=2)
    ax.axvline(x=phase1_end + 0.5, color="purple", linestyle="--", label="Fine-Tuning Begins")
    ax.set_title(title)
    ax.set_xlabel("Epochs")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix – Brain Tumor Classification Results", fontweight="bold")
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()

Classical CNN vs Hybrid QCNN Comparision


In [ ]:
import os
import random
import pathlib
import logging
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

from tensorflow.keras import Input
from tensorflow.keras.layers import (Dense, GlobalAveragePooling2D, Dropout,
                                     BatchNormalization, Conv2D, Reshape,
                                     AveragePooling2D, Layer, Flatten, Concatenate)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("======================================================")
print("  QCNN vs CNN NOISE ROBUSTNESS EXPERIMENT SUITE")
print("======================================================\n")

# =========================================================================
#  PART 1: SETUP & SMART DATASET FINDER
# =========================================================================
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

NUM_CLASSES = 4
BATCH_SIZE = 32

BASE_DIR = "/kaggle/input"
all_paths = []
all_labels = []

# The four standard classes for this dataset
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

print("Scanning Kaggle input directory for images...")

# Smart Finder: Walks through all folders automatically
for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        # Accept any standard image format
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(root, file)
            
            # Identify the class by looking at the folder the image is inside
            folder_name = os.path.basename(os.path.dirname(full_path)).lower()
            
            # Match the folder name to our 4 classes
            for i, class_name in enumerate(class_names):
                # Handles datasets that spell it "no_tumor" instead of "notumor"
                search_name = "no_tumor" if class_name == "notumor" else class_name
                
                if class_name in folder_name or search_name in folder_name:
                    all_paths.append(full_path)
                    all_labels.append(i)
                    break

if not all_paths:
    print(f"\nAvailable folders in {BASE_DIR}:", os.listdir(BASE_DIR))
    raise ValueError("Still couldn't find the images! Check the folder names printed above.")

# Split into 80% Train, 20% Temp (Val + Test)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.20, random_state=42, stratify=all_labels
)

# Split Temp into 10% Val, 10% Test
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

print(f"✅ Success! Dataset Loaded: {len(train_paths)} Train | {len(val_paths)} Val | {len(test_paths)} Test")

# ─────────────────────────────────────────────────────────────────────────
#  2. DATA GENERATORS & DYNAMIC NOISE INJECTION
# ─────────────────────────────────────────────────────────────────────────
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def get_label(file_path):
    parts = tf.strings.split(file_path, os.path.sep)
    return tf.argmax(parts[-2] == class_names)

def process_img(file_path, noise_level=0.0):
    label = get_label(file_path)
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    
    # Scale to [-1, 1] for MobileNetV2
    img = preprocess_input(img)
    
    # Inject Gaussian Noise dynamically (Only applied if noise_level > 0)
    if noise_level > 0.0:
        noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=noise_level, dtype=tf.float32)
        img = img + noise
        img = tf.clip_by_value(img, -1.0, 1.0) # Keep within valid bounds
        
    return img, tf.one_hot(label, NUM_CLASSES)

def create_dataset(paths, noise_level=0.0, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices([str(p) for p in paths])
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths))
    
    # Pass the noise parameter into the mapping function
    ds = ds.map(lambda x: process_img(x, noise_level), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ─────────────────────────────────────────────────────────────────────────
#  3. QUANTUM SIMULATOR & MODEL ARCHITECTURES
# ─────────────────────────────────────────────────────────────────────────
@tf.keras.utils.register_keras_serializable()
class TFQuantumSimulationLayer(Layer):
    """Optimized batch-level Quantum Simulator"""
    def __init__(self, n_qubits=4, n_layers=3, **kwargs) -> None:
        super().__init__(**kwargs)
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.state_size = 2 ** n_qubits
        self.cnot_perms = [self._get_cnot_perm(q, q+1) for q in range(self.n_qubits - 1)]
        self.z_masks = []
        for q in range(self.n_qubits):
            mask = [-1.0 if (idx & (1 << (self.n_qubits - 1 - q))) != 0 else 1.0 for idx in range(self.state_size)]
            self.z_masks.append(tf.constant(mask, dtype=tf.float32))

    def _get_cnot_perm(self, control, target):
        perm = [i ^ (1 << (self.n_qubits - 1 - target)) if (i & (1 << (self.n_qubits - 1 - control))) else i for i in range(self.state_size)]
        return tf.constant(perm, dtype=tf.int32)

    def build(self, input_shape) -> None:
        self.params = self.add_weight(name="quantum_weights", shape=(self.n_layers, self.n_qubits, 3), initializer="random_uniform", trainable=True)
        super().build(input_shape)

    def _get_rotation_matrices(self, phi, theta, omega):
        phi, theta, omega = tf.cast(phi, tf.complex64), tf.cast(theta, tf.complex64), tf.cast(omega, tf.complex64)
        cos_t, sin_t = tf.cos(theta / 2.0), tf.sin(theta / 2.0)
        u00 = tf.exp(-1j * (phi + omega) / 2.0) * cos_t
        u01 = -tf.exp(-1j * (phi - omega) / 2.0) * sin_t
        u10 = tf.exp(1j * (phi - omega) / 2.0) * sin_t
        u11 = tf.exp(1j * (phi + omega) / 2.0) * cos_t
        return tf.stack([tf.stack([u00, u01], axis=-1), tf.stack([u10, u11], axis=-1)], axis=-2)

    def call(self, inputs):
        x = tf.clip_by_value(inputs * np.pi, -np.pi, np.pi)
        batch_size = tf.shape(tf.reshape(x, (-1, self.n_qubits)))[0]
        batch_flat = tf.reshape(x, (-1, self.n_qubits))

        state = tf.concat([tf.ones((batch_size, 1), dtype=tf.complex64), tf.zeros((batch_size, self.state_size - 1), dtype=tf.complex64)], axis=1)

        def apply_gate(st, U, qubit):
            st_reshaped = tf.reshape(st, [-1, 2**qubit, 2, 2**(self.n_qubits - 1 - qubit)])
            return tf.reshape(tf.einsum('boi, blir -> blor', U, st_reshaped), [-1, self.state_size])

        for l in range(self.n_layers):
            for q in range(self.n_qubits):
                state = apply_gate(state, self._get_rotation_matrices(batch_flat[:, q], batch_flat[:, q], batch_flat[:, q]), q)
            for q in range(self.n_qubits):
                p = self.params[l, q]
                U = self._get_rotation_matrices(p[0] * tf.ones([batch_size], dtype=tf.float32), p[1] * tf.ones([batch_size], dtype=tf.float32), p[2] * tf.ones([batch_size], dtype=tf.float32))
                state = apply_gate(state, U, q)
            for q in range(self.n_qubits - 1):
                state = tf.gather(state, self.cnot_perms[q], axis=1)

        state_density = tf.abs(state) ** 2
        return tf.reshape(tf.stack([tf.reduce_sum(state_density * self.z_masks[q], axis=1) for q in range(self.n_qubits)], axis=1), (tf.shape(x)[0], tf.shape(x)[1], self.n_qubits))


def build_pure_cnn(num_classes):
    """Classical Baseline (MobileNetV2 only)"""
    inputs = Input(shape=(224, 224, 3))
    base_model = tf.keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

def build_hybrid_qcnn(num_classes):
    """Proposed Model (MobileNetV2 + Quantum Branch)"""
    inputs = Input(shape=(224, 224, 3))
    
    # Branch 1: Classical
    base_model = tf.keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    x_mb = base_model(inputs, training=False)
    x_mb = GlobalAveragePooling2D()(x_mb)
    
    # Branch 2: Quantum
    x_q = AveragePooling2D(pool_size=56, strides=56)(inputs)
    x_q = Conv2D(4, kernel_size=1, activation="relu")(x_q)
    x_q = Reshape((-1, 4))(x_q)
    x_q = TFQuantumSimulationLayer(4, 3)(x_q)
    x_q = Flatten()(x_q)
    x_q = Dense(32, activation="relu")(x_q)
    
    # Fusion
    x = Concatenate()([x_mb, x_q])
    x = Dense(512, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# ─────────────────────────────────────────────────────────────────────────
#  4. STRESS TEST EXECUTION LOOP WITH ANTI-OVERTRAINING SAFEGUARDS
# ─────────────────────────────────────────────────────────────────────────
NOISE_LEVELS = [0.0, 0.005, 0.05, 0.10, 0.15]

# Maximum upper bound for training per noise run
STRESS_TEST_EPOCHS = 30 

# Helper function to generate fresh callbacks per model training run
def get_anti_overtrain_callbacks():
    return [
        # Stop training when validation loss stops improving for 5 consecutive epochs
        # Automatically reverts weights to the best epoch achieved
        EarlyStopping(
            monitor="val_loss", 
            patience=5, 
            restore_best_weights=True, 
            verbose=1
        ),
        # Reduce learning rate if validation loss plateaus to fine-tune convergence
        ReduceLROnPlateau(
            monitor="val_loss", 
            factor=0.3, 
            patience=2, 
            min_lr=1e-7, 
            verbose=1
        )
    ]

cnn_accuracies = []
qcnn_accuracies = []

# Clean test and validation sets (Noise = 0)
val_ds = create_dataset(val_paths, noise_level=0.0, shuffle=False)
test_ds = create_dataset(test_paths, noise_level=0.0, shuffle=False)

for noise in NOISE_LEVELS:
    print(f"\n{'='*60}")
    print(f" TRAINING MODELS WITH {noise * 100}% GAUSSIAN NOISE")
    print(f"{'='*60}")
    
    # Create a noisy training dataset specific to this loop iteration
    train_ds = create_dataset(train_paths, noise_level=noise, shuffle=True)
    
    print("\n--- Training Pure Classical CNN ---")
    cnn = build_pure_cnn(NUM_CLASSES)
    cnn.fit(
        train_ds, 
        validation_data=val_ds, 
        epochs=STRESS_TEST_EPOCHS, 
        callbacks=get_anti_overtrain_callbacks(),
        verbose=1
    )
    
    print("\n--- Evaluating CNN on Clean Test Set (Best Weights Restored) ---")
    _, cnn_acc = cnn.evaluate(test_ds, verbose=0)
    cnn_accuracies.append(cnn_acc)
    print(f"Classical CNN Accuracy: {cnn_acc * 100:.2f}%")
    
    print("\n--- Training Hybrid QCNN ---")
    qcnn = build_hybrid_qcnn(NUM_CLASSES)
    qcnn.fit(
        train_ds, 
        validation_data=val_ds, 
        epochs=STRESS_TEST_EPOCHS, 
        callbacks=get_anti_overtrain_callbacks(),
        verbose=1
    )
    
    print("\n--- Evaluating QCNN on Clean Test Set (Best Weights Restored) ---")
    _, qcnn_acc = qcnn.evaluate(test_ds, verbose=0)
    qcnn_accuracies.append(qcnn_acc)
    print(f"Hybrid QCNN Accuracy: {qcnn_acc * 100:.2f}%")

# ─────────────────────────────────────────────────────────────────────────
#  5. RESULTS VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.plot([n * 100 for n in NOISE_LEVELS], [a * 100 for a in cnn_accuracies], marker='o', linestyle='-', label='Classical CNN', color='red')
plt.plot([n * 100 for n in NOISE_LEVELS], [a * 100 for a in qcnn_accuracies], marker='s', linestyle='-', label='Hybrid QCNN', color='blue')

plt.title('Quantum Advantage: Model Robustness to Training Noise', fontsize=14, fontweight='bold')
plt.xlabel('Noise Percentage Added to Training Data (%)', fontsize=12)
plt.ylabel('Test Accuracy on Clean Data (%)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks([n * 100 for n in NOISE_LEVELS])

# Add exact percentage labels to the chart points
for i, (cnn_a, qcnn_a) in enumerate(zip(cnn_accuracies, qcnn_accuracies)):
    plt.text(NOISE_LEVELS[i]*100, cnn_a*100 - 1.5, f"{cnn_a*100:.1f}%", color='red', ha='center')
    plt.text(NOISE_LEVELS[i]*100, qcnn_a*100 + 1, f"{qcnn_a*100:.1f}%", color='blue', ha='center')

plt.tight_layout()
plt.show()

Test of Models

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import Xception, ResNet50V2, InceptionV3

# =========================================================================
#  PART 1: SETUP & SMART DATASET FINDER
# =========================================================================
NUM_CLASSES = 4
BATCH_SIZE = 32
BASE_DIR = "/kaggle/input"

all_paths = []
all_labels = []

class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

print("Scanning Kaggle input directory for images...")

for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(root, file)
            folder_name = os.path.basename(os.path.dirname(full_path)).lower()
            
            for i, class_name in enumerate(class_names):
                search_name = "no_tumor" if class_name == "notumor" else class_name
                if class_name in folder_name or search_name in folder_name:
                    all_paths.append(full_path)
                    all_labels.append(i)
                    break

if not all_paths:
    print(f"\nAvailable folders in {BASE_DIR}:", os.listdir(BASE_DIR))
    raise ValueError("Still couldn't find the images! Check the folder names printed above.")

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.20, random_state=42, stratify=all_labels
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

print(f"✅ Success! Dataset Loaded: {len(train_paths)} Train | {len(val_paths)} Val | {len(test_paths)} Test")


# =========================================================================
#  PART 2: TENSORFLOW DATA PIPELINE & NOISE INJECTION
# =========================================================================
def add_gaussian_noise(image, noise_level):
    noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=noise_level, dtype=tf.float32)
    image = image + noise
    return tf.clip_by_value(image, 0.0, 1.0) 

def process_path(file_path, label, noise_level=0.0):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = img / 255.0 
    
    if noise_level > 0.0:
        img = add_gaussian_noise(img, noise_level)
        
    label = tf.one_hot(label, depth=NUM_CLASSES)
    return img, label

def create_dataset(file_paths, labels, noise_level=0.0, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(file_paths))
    ds = ds.map(lambda x, y: process_path(x, y, noise_level), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def get_anti_overtrain_callbacks():
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    return [early_stop, reduce_lr]


# =========================================================================
#  PART 3: ARCHITECTURE BUILDERS
# =========================================================================
def build_xception(num_classes):
    inputs = Input(shape=(224, 224, 3))
    base_model = Xception(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False 
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

def build_resnet(num_classes):
    inputs = Input(shape=(224, 224, 3))
    base_model = ResNet50V2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

def build_inception(num_classes):
    inputs = Input(shape=(224, 224, 3))
    base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


# =========================================================================
#  PART 4: 5-LEVEL NOISE ROBUSTNESS STRESS TEST
# =========================================================================
NOISE_LEVELS = [0.0, 0.005, 0.05, 0.10, 0.15]
STRESS_TEST_EPOCHS = 30 

val_ds = create_dataset(val_paths, val_labels, noise_level=0.0, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, noise_level=0.0, shuffle=False)

results = {
    "Xception": [],
    "ResNet50V2": [],
    "InceptionV3": []
}

for noise in NOISE_LEVELS:
    noise_pct = noise * 100
    print(f"\n{'='*65}")
    print(f" EVALUATING MODELS AT {noise_pct:.1f}% GAUSSIAN NOISE")
    print(f"{'='*65}")
    
    train_ds = create_dataset(train_paths, train_labels, noise_level=noise, shuffle=True)
    
    models_to_test = {
        "Xception": build_xception(NUM_CLASSES),
        "ResNet50V2": build_resnet(NUM_CLASSES),
        "InceptionV3": build_inception(NUM_CLASSES)
    }
    
    for name, model in models_to_test.items():
        print(f"\n--- Training {name} ({noise_pct:.1f}% Noise) ---")
        
        model.fit(
            train_ds, 
            validation_data=val_ds, 
            epochs=STRESS_TEST_EPOCHS, 
            callbacks=get_anti_overtrain_callbacks(),
            verbose=1
        )
        
        _, acc = model.evaluate(test_ds, verbose=0)
        acc_percentage = round(acc * 100, 2)
        
        results[name].append(acc_percentage)
        print(f"\n>>> [{name}] @ {noise_pct:.1f}% Noise -> Clean Test Acc: {acc_percentage}%\n")


# =========================================================================
#  PART 5: PRINT FORMATTED OUTPUT
# =========================================================================
print("\n" + "="*65)
print(" FINAL REAL DATA ARRAYS ACROSS ALL 5 NOISE LEVELS")
print("="*65)
print(f"noise_levels = {NOISE_LEVELS}")
for name, acc_list in results.items():
    print(f"{name.lower()}_accuracies = {acc_list}")

In [ ]:
!pip install pennylane

In [ ]:
import os
import numpy as np
import tensorflow as tf
import pennylane as qml
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =========================================================================
#  PART 1: SETUP & DATASET PIPELINE
# =========================================================================
NUM_CLASSES = 4
BATCH_SIZE = 32
BASE_DIR = "/kaggle/input"
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

all_paths, all_labels = [], []
for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(root, file)
            folder_name = os.path.basename(os.path.dirname(full_path)).lower()
            for i, c_name in enumerate(class_names):
                search_name = "no_tumor" if c_name == "notumor" else c_name
                if c_name in folder_name or search_name in folder_name:
                    all_paths.append(full_path)
                    all_labels.append(i)
                    break

train_paths, temp_paths, train_labels, temp_labels = train_test_split(all_paths, all_labels, test_size=0.20, random_state=42, stratify=all_labels)
val_paths, test_paths, val_labels, test_labels = train_test_split(temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels)

def process_path(file_path, label, noise_level=0.0):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224]) / 255.0 
    if noise_level > 0.0:
        noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=noise_level, dtype=tf.float32)
        img = tf.clip_by_value(img + noise, 0.0, 1.0)
    return img, tf.one_hot(label, depth=NUM_CLASSES)

def create_dataset(file_paths, labels, noise_level=0.0, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    if shuffle: ds = ds.shuffle(buffer_size=len(file_paths))
    return ds.map(lambda x, y: process_path(x, y, noise_level), num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def get_anti_overtrain_callbacks():
    return [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)]


# =========================================================================
#  PART 2: FIXED GRADIENT-SAFE QUANTUM ENGINE
# =========================================================================
class QuantumLayer(tf.keras.layers.Layer):
    """Custom Keras Layer that perfectly bridges PennyLane float64 and TensorFlow float32"""
    def __init__(self, qnode, weight_shapes, output_dim, **kwargs):
        super(QuantumLayer, self).__init__(**kwargs)
        self.qnode = qnode
        self.weight_shapes = weight_shapes
        self.output_dim = output_dim
        self.qnode_weights = {}

    def build(self, input_shape):
        for name, shape in self.weight_shapes.items():
            self.qnode_weights[name] = self.add_weight(
                name=name, shape=shape,
                initializer=tf.keras.initializers.RandomUniform(minval=-np.pi, maxval=np.pi),
                trainable=True, dtype=tf.float32
            )
        super(QuantumLayer, self).build(input_shape)

    def call(self, inputs):
        def run_qnode(x):
            x_f32 = tf.cast(x, tf.float32)
            res = self.qnode(x_f32, self.qnode_weights["weights"])
            if isinstance(res, (list, tuple)): res = tf.stack(res)
            # Safely extract real part so TensorFlow gradient tape doesn't crash
            return tf.cast(tf.math.real(res), tf.float32) 
        
        return tf.map_fn(run_qnode, inputs, fn_output_signature=tf.float32)
    
    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.output_dim)


# =========================================================================
#  PART 3: 8-QUBIT HYBRID ARCHITECTURES (High Accuracy Config)
# =========================================================================
n_qubits = 8

# ---------------------------------------------------------
# A. Advanced QuanvNN (Custom CNN Base -> 8-Qubit Filter)
# ---------------------------------------------------------
dev_quanv = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev_quanv, interface="tf")
def quanv_circuit(inputs, weights):
    for i in range(n_qubits): qml.RY(inputs[i] * np.pi, wires=i)
    for i in range(n_qubits): qml.RX(weights[i], wires=i)
    for i in range(n_qubits - 1): qml.CNOT(wires=[i, i + 1])
    qml.CNOT(wires=[n_qubits - 1, 0])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def build_advanced_quanv(num_classes):
    qlayer = QuantumLayer(quanv_circuit, {"weights": (n_qubits,)}, output_dim=n_qubits)
    inputs = Input(shape=(224, 224, 3))
    x = Conv2D(32, (3, 3), activation="relu")(inputs)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation="relu")(x)
    x = GlobalAveragePooling2D()(x)
    
    x = Dense(64, activation="relu")(x) # Funnel down
    x_q = Dense(n_qubits, activation="tanh")(x) # Tanh scales inputs for quantum gates
    x_q = qlayer(x_q)
    
    x = Dense(128, activation="relu")(x_q)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# ---------------------------------------------------------
# B. Dense VQC Hybrid (ResNet50V2 Base -> 8-Qubit Bottleneck)
# ---------------------------------------------------------
dev_dense = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev_dense, interface="tf")
def dense_vqc_circuit(inputs, weights):
    for i in range(n_qubits): qml.RY(inputs[i] * np.pi, wires=i)
    for l in range(2): # 2 Strongly Entangling Layers
        for i in range(n_qubits): qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
        for i in range(n_qubits - 1): qml.CNOT(wires=[i, i + 1])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def build_dense_hybrid(num_classes):
    qlayer = QuantumLayer(dense_vqc_circuit, {"weights": (2, n_qubits, 3)}, output_dim=n_qubits)
    base_model = ResNet50V2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    
    inputs = Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    
    x = Dense(128, activation="relu")(x) # Graceful dimensionality reduction from 2048 to 128
    x = Dense(n_qubits, activation="tanh")(x) # Down to 8 qubits
    x = qlayer(x)
    
    x = Dense(64, activation="relu")(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# ---------------------------------------------------------
# C. Dressed Quantum Circuit (CNN Base -> 8-Qubit Ring)
# ---------------------------------------------------------
dev_dressed = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev_dressed, interface="tf")
def dressed_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
        qml.RZ(inputs[i] * np.pi, wires=i)
    for i in range(n_qubits):
        qml.CRZ(weights[i], wires=[i, (i + 1) % n_qubits])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def build_dressed_hybrid(num_classes):
    qlayer = QuantumLayer(dressed_circuit, {"weights": (n_qubits,)}, output_dim=n_qubits)
    inputs = Input(shape=(224, 224, 3))
    x = Conv2D(64, (3, 3), activation="relu")(inputs)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation="relu")(x)
    x = GlobalAveragePooling2D()(x)
    
    x = Dense(n_qubits, activation="tanh")(x)
    x = qlayer(x)
    
    x = Dense(64, activation="relu")(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


# =========================================================================
#  PART 4: 5-LEVEL NOISE STRESS TEST (HYBRIDS ONLY)
# =========================================================================
NOISE_LEVELS = [0.0, 0.005, 0.05, 0.10, 0.15]
STRESS_TEST_EPOCHS = 30 # Let early stopping do its job

val_ds = create_dataset(val_paths, val_labels, noise_level=0.0, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, noise_level=0.0, shuffle=False)

hybrid_results = {"Advanced_Quanv": [], "Dense_VQC_ResNet": [], "Dressed_QNN": []}

for noise in NOISE_LEVELS:
    noise_pct = noise * 100
    print(f"\n{'='*75}")
    print(f" EVALUATING 8-QUBIT HYBRID MODELS AT {noise_pct:.1f}% GAUSSIAN NOISE")
    print(f"{'='*75}")
    
    train_ds = create_dataset(train_paths, train_labels, noise_level=noise, shuffle=True)
    
    hybrid_models = {
        "Advanced_Quanv": build_advanced_quanv(NUM_CLASSES),
        "Dense_VQC_ResNet": build_dense_hybrid(NUM_CLASSES),
        "Dressed_QNN": build_dressed_hybrid(NUM_CLASSES)
    }
    
    for name, model in hybrid_models.items():
        print(f"\n--- Training {name} ({noise_pct:.1f}% Noise) ---")
        
        model.fit(
            train_ds, validation_data=val_ds, 
            epochs=STRESS_TEST_EPOCHS, 
            callbacks=get_anti_overtrain_callbacks(),
            verbose=1
        )
        
        _, acc = model.evaluate(test_ds, verbose=0)
        acc_percentage = round(acc * 100, 2)
        hybrid_results[name].append(acc_percentage)
        
        print(f"\n>>> [{name}] @ {noise_pct:.1f}% Noise -> Clean Test Acc: {acc_percentage}%\n")
        
        

print("\n" + "="*75)
print(" FINAL 8-QUBIT HYBRID MODEL ARRAYS ACROSS ALL 5 NOISE LEVELS")
print("="*75)
print(f"noise_levels = {NOISE_LEVELS}")
for name, acc_list in hybrid_results.items():
    print(f"{name.lower()}_accuracies = {acc_list}")

In [ ]:
!pip install pennylane

In [1]:
import os
import numpy as np
import tensorflow as tf
import pennylane as qml
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =========================================================================
#  PART 1: DATASET PIPELINE & PRE-EXTRACTION SETUP
# =========================================================================
NUM_CLASSES = 4
BATCH_SIZE = 64
BASE_DIR = "/kaggle/input"
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

all_paths, all_labels = [], []
for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(root, file)
            folder_name = os.path.basename(os.path.dirname(full_path)).lower()
            for i, c_name in enumerate(class_names):
                search_name = "no_tumor" if c_name == "notumor" else c_name
                if c_name in folder_name or search_name in folder_name:
                    all_paths.append(full_path)
                    all_labels.append(i)
                    break

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.20, random_state=42, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

def process_path(file_path, label, noise_level=0.0):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224]) / 255.0 
    if noise_level > 0.0:
        noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=noise_level, dtype=tf.float32)
        img = tf.clip_by_value(img + noise, 0.0, 1.0)
    return img, label

def create_dataset(file_paths, labels, noise_level=0.0):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    return ds.map(lambda x, y: process_path(x, y, noise_level), num_parallel_calls=tf.data.AUTOTUNE)\
             .batch(BATCH_SIZE)\
             .prefetch(tf.data.AUTOTUNE)

def get_anti_overtrain_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    ]

# Build Frozen Backbone Feature Extractor (Outputs 1D Global Pooled Vectors)
def build_extractor():
    base = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base.trainable = False
    inputs = Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    outputs = tf.keras.layers.GlobalAveragePooling2D()(x)
    return Model(inputs=inputs, outputs=outputs)

print("Initializing Frozen Feature Extractor...")
extractor = build_extractor()


# =========================================================================
#  PART 2: OPTIMIZED CUSTOM QUANTUM LAYER (Keras 3 / Modern TF Compatible)
# =========================================================================
class QuantumLayer(tf.keras.layers.Layer):
    """Custom Layer bypassing the removed qml.qnn and the TF tensordot broadcasting bug"""
    def __init__(self, qnode, weight_shapes, output_dim, **kwargs):
        super(QuantumLayer, self).__init__(**kwargs)
        self.qnode = qnode
        self.weight_shapes = weight_shapes
        self.output_dim = output_dim
        self.qnode_weights = {}

    def build(self, input_shape):
        for name, shape in self.weight_shapes.items():
            self.qnode_weights[name] = self.add_weight(
                name=name, shape=shape,
                initializer=tf.keras.initializers.RandomUniform(minval=-np.pi, maxval=np.pi),
                trainable=True, 
                dtype=tf.float32
            )
        super(QuantumLayer, self).build(input_shape)

    def call(self, inputs):
        # We use a parallelized map_fn. This avoids the PennyLane dimension mismatch bug 
        # while processing the batch asynchronously in the TF graph.
        def _qnode_step(x):
            res = self.qnode(x, self.qnode_weights["weights"])
            if isinstance(res, (list, tuple)): 
                res = tf.stack(res)
            return tf.cast(tf.math.real(res), tf.float32)

        # fn_output_signature ensures TF compiles the shape correctly
        batch_res = tf.map_fn(
            _qnode_step, 
            inputs, 
            fn_output_signature=tf.TensorSpec(shape=(self.output_dim,), dtype=tf.float32),
            parallel_iterations=32 # Maximizes parallel execution speed
        )
        return batch_res

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.output_dim)


# =========================================================================
#  PART 3: 8-QUBIT QUANTUM CIRCUIT & HEAD MODEL
# =========================================================================
n_qubits = 8

# DenseNet Circuit
dev_densenet = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev_densenet, interface="tf", diff_method="backprop")
def densenet_vqc_circuit(inputs, weights):
    qml.AngleEmbedding(inputs * np.pi, wires=range(n_qubits), rotation='Y')
    for l in range(2): 
        for i in range(n_qubits): 
            qml.RX(weights[l, i, 0], wires=i)
            qml.RZ(weights[l, i, 1], wires=i)
        for i in range(n_qubits): 
            qml.CZ(wires=[i, (i + 1) % n_qubits])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def build_head_model(feature_dim):
    w_shape = (2, n_qubits, 2)
    # Using our custom QuantumLayer
    qlayer = QuantumLayer(densenet_vqc_circuit, {"weights": w_shape}, output_dim=n_qubits)
    
    inputs = Input(shape=(feature_dim,))
    x = Dense(128, activation="relu")(inputs)
    x = Dense(n_qubits, activation="tanh")(x)
    x = qlayer(x)
    x = Dense(64, activation="relu")(x)
    outputs = Dense(NUM_CLASSES, activation="softmax")(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


# =========================================================================
#  PART 4: FAST NOISE STRESS TEST LOOP
# =========================================================================
NOISE_LEVELS = [0.0, 0.005, 0.05, 0.10, 0.15]
STRESS_TEST_EPOCHS = 30 

val_ds_raw = create_dataset(val_paths, val_labels, noise_level=0.0)
test_ds_raw = create_dataset(test_paths, test_labels, noise_level=0.0)

val_y = tf.one_hot(val_labels, depth=NUM_CLASSES)
test_y = tf.one_hot(test_labels, depth=NUM_CLASSES)

print("\nPre-extracting clean Validation and Test features...")
val_feats = extractor.predict(val_ds_raw, verbose=0)
test_feats = extractor.predict(test_ds_raw, verbose=0)

hybrid_results = []

for noise in NOISE_LEVELS:
    noise_pct = noise * 100
    print(f"\n{'='*75}\n EVALUATING FAST 8-QUBIT HYBRID MODEL AT {noise_pct:.1f}% GAUSSIAN NOISE\n{'='*75}")
    
    train_ds_raw = create_dataset(train_paths, train_labels, noise_level=noise)
    train_y = tf.one_hot(train_labels, depth=NUM_CLASSES)
    
    # Extract features for noisy training images ONCE per noise level
    print(f"Pre-extracting features for {noise_pct:.1f}% noise training set...")
    train_feats = extractor.predict(train_ds_raw, verbose=0)
    
    print(f"\n--- Training DenseNet_VQC_Hybrid ({noise_pct:.1f}% Noise) ---")
    
    feature_dim = train_feats.shape[1]
    model = build_head_model(feature_dim)
    
    # Train ONLY the small quantum head on pre-extracted feature vectors
    model.fit(
        train_feats, train_y,
        validation_data=(val_feats, val_y),
        batch_size=32,
        epochs=STRESS_TEST_EPOCHS,
        callbacks=get_anti_overtrain_callbacks(),
        verbose=1
    )
    
    _, acc = model.evaluate(test_feats, test_y, verbose=0)
    acc_percentage = round(acc * 100, 2)
    hybrid_results.append(acc_percentage)
    
    print(f"\n>>> [DenseNet_VQC_Hybrid] @ {noise_pct:.1f}% Noise -> Clean Test Acc: {acc_percentage}%\n")
    
    # FIX: Ensure the save name ends exactly with .weights.h5 for Keras 3 compatibility
    save_name = f"DenseNet_VQC_Hybrid_8Qubit_weights_at_{noise_pct}_noise.weights.h5"
    model.save_weights(save_name)
  

# =========================================================================
#  PART 5: SUMMARY RESULTS
# =========================================================================
print("\n" + "="*75)
print(" FINAL 8-QUBIT HYBRID MODEL ARRAYS ACROSS ALL 5 NOISE LEVELS")
print("="*75)
print(f"noise_levels = {NOISE_LEVELS}")
print(f"densenet_vqc_hybrid_accuracies = {hybrid_results}")

Initializing Frozen Feature Extractor...


I0000 00:00:1785436366.243665    3463 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785436366.246063    3463 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



Pre-extracting clean Validation and Test features...


I0000 00:00:1785436391.247023    3509 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



 EVALUATING FAST 8-QUBIT HYBRID MODEL AT 0.0% GAUSSIAN NOISE
Pre-extracting features for 0.0% noise training set...

--- Training DenseNet_VQC_Hybrid (0.0% Noise) ---
Epoch 1/30


/tmp/ipykernel_3463/207364038.py:100: PennyLaneDeprecationWarning: Support for the TensorFlow interface is deprecated and will be removed in v0.44. Future versions of PennyLane are not guaranteed to work with TensorFlow. Please migrate your workflows to JAX or Pytorch to benefit from enhanced support and features.
  res = self.qnode(x, self.qnode_weights["weights"])


180/180 ━━━━━━━━━━━━━━━━━━━━ 59s 125ms/step - accuracy: 0.3708 - loss: 1.3312 - val_accuracy: 0.5972 - val_loss: 1.2612 - learning_rate: 1.0000e-04
Epoch 2/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 11s 60ms/step - accuracy: 0.7078 - loss: 1.1719 - val_accuracy: 0.7458 - val_loss: 1.0996 - learning_rate: 1.0000e-04
Epoch 3/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accuracy: 0.8113 - loss: 1.0015 - val_accuracy: 0.8111 - val_loss: 0.9381 - learning_rate: 1.0000e-04
Epoch 4/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accuracy: 0.8528 - loss: 0.8325 - val_accuracy: 0.8292 - val_loss: 0.7913 - learning_rate: 1.0000e-04
Epoch 5/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accuracy: 0.8816 - loss: 0.6691 - val_accuracy: 0.8542 - val_loss: 0.6371 - learning_rate: 1.0000e-04
Epoch 6/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accuracy: 0.8953 - loss: 0.5327 - val_accuracy: 0.8444 - val_loss: 0.5517 - learning_rate: 1.0000e-04
Epoch 7/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accura